# 06 — News Intelligence & Sentiment

Phase 6 of the Indian Equity AI project plan.

Pipeline: news ingestion -> dedup/entity resolution -> FinBERT sentiment ->
event classification -> historical event-impact estimation -> daily (date, symbol)
news feature table.

**Data-source limitation:** Yahoo Finance's free news feed only returns a small,
recent rolling window per ticker (~10 items, last few days) -- not a multi-year
historical archive. The pipeline below is fully real (no synthetic/fabricated
news), but the event-impact statistics at the end are based on a small, recent
sample and should be read as a methodology demonstration, not a statistically
reliable estimate. A production system would need a licensed news provider
with historical coverage (see Indian_Equity_AI_Project_Plan.md Section 5).

In [1]:
# IMPORTANT: torch must be imported before pandas/pyarrow anywhere in this
# process, or its native DLL load fails on Windows (see src/news/sentiment.py
# module docstring for details). Keep this as the very first import.
import torch  # noqa: F401

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.news.ingestion import fetch_universe_news
from src.news.dedup import deduplicate_news, resolve_entity_mentions
from src.news.sentiment import add_sentiment, load_finbert_pipeline
from src.news.events import add_event_types
from src.news.impact import compute_event_occurrence_returns, explode_event_types, summarize_event_impact
from src.news.features import aggregate_daily_news_features

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
from src.config import UNIVERSE  # Phase 14: centralized, now 50 stocks

## Step 1: News ingestion

In [3]:
news = fetch_universe_news(UNIVERSE)
print("Raw articles fetched:", len(news))
news["symbol"].value_counts()

Raw articles fetched: 0


Series([], Name: count, dtype: int64)

## Step 2: Deduplication and entity resolution

In [4]:
news = deduplicate_news(news)
news = resolve_entity_mentions(news)
print("After dedup:", len(news))
print("Genuinely mention the company:", news["mentions_company"].sum(), "/", len(news))

news = news[news["mentions_company"]].reset_index(drop=True)
news[["symbol", "headline", "published_timestamp"]]

After dedup: 0
Genuinely mention the company: 0 / 0


,symbol,headline,published_timestamp


## Step 3: FinBERT financial sentiment

In [5]:
finbert = load_finbert_pipeline()
news = add_sentiment(news, finbert)
news[["symbol", "headline", "positive_probability", "neutral_probability", "negative_probability"]]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

,symbol,headline,positive_probability,neutral_probability,negative_probability


## Step 4: Event classification

In [6]:
news = add_event_types(news)
news[["symbol", "headline", "event_types"]]

,symbol,headline,event_types


## Step 5: Historical event-impact analysis

How has each event type historically moved the stock at +1d / +5d / +20d?
(See caveat at the top of this notebook re: small sample size.)

In [7]:
prices = pd.read_parquet(DATA_PROCESSED / "prices_daily.parquet")[["date", "symbol", "close"]]

events_long = explode_event_types(news)
print("Event occurrences (news x event_type pairs):", len(events_long))

occurrence_returns = compute_event_occurrence_returns(events_long, prices, horizons=(1, 5, 20))
impact_summary = summarize_event_impact(occurrence_returns, horizons=(1, 5, 20))
impact_summary

Event occurrences (news x event_type pairs): 0


,event_type,occurrences,mean_return_1d,mean_return_5d,mean_return_20d


## Step 6: Aggregate to a daily (date, symbol) news feature table

In [8]:
daily_news_features = aggregate_daily_news_features(news)
daily_news_features

,date,symbol,news_count,sentiment_mean,sentiment_std,positive_news_ratio,negative_news_ratio


## Step 7: Save

In [9]:
news.drop(columns=["event_types"]).assign(
    event_types=news["event_types"].apply(lambda types: ",".join(types))
).to_csv(DATA_PROCESSED / "news_scored.csv", index=False)

daily_news_features.to_parquet(DATA_PROCESSED / "news_daily_features.parquet", index=False)
impact_summary.to_csv(DATA_PROCESSED / "news_event_impact_summary.csv", index=False)

print("Saved:", DATA_PROCESSED / "news_scored.csv")
print("Saved:", DATA_PROCESSED / "news_daily_features.parquet")
print("Saved:", DATA_PROCESSED / "news_event_impact_summary.csv")

Saved: D:\Practise\Stock Market Analysis\data\processed\news_scored.csv
Saved: D:\Practise\Stock Market Analysis\data\processed\news_daily_features.parquet
Saved: D:\Practise\Stock Market Analysis\data\processed\news_event_impact_summary.csv
